In [ ]:
# pip install -U git+https://github.com/huggingface/transformers.git
# pip install -U triton triton_kernels accelerate uv
# git clone https://github.com/eelbaz/dgx-spark-vllm-setup.git
# cd dgx-spark-vllm-setup
# ./install.sh
# cd dgx-spark-vllm-setup/vllm-install/vllm
# grep -rl "flashinfer-python==0.4.1" . | xargs sed -i 's/flashinfer-python==0.4.1/flashinfer-python>=0.4.2/g'
# uv pip install --system setuptools_scm ninja packaging wheel scons
# uv pip install --system -e . --no-build-isolation

# vllm serve openai/gpt-oss-20b --trust-remote-code --gpu-memory-utilization 0.3 --port 58080
# vllm serve Qwen/Qwen3.5-35B-A3B --trust-remote-code --gpu-memory-utilization 0.9 --port 58080
# vllm serve Qwen/Qwen3.5-35B-A3B-GPTQ-Int4 --quantization gptq --dtype float16 --trust-remote-code --gpu-memory-utilization 0.25 --port 58080

In [2]:
!pip install langchain langchain_core langchain_ollama

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser

In [2]:
llm = ChatOllama(
    # model="gemma3n:e2b",
    model="qwen3.5:35b",
    # model_kwargs={"think": False},
    # think=False,          # Ollama 버전
    reasoning=True,      # ChatOllama 버전
    temperature=0.0
)

In [3]:
import base64
# 이미지를 base64로 인코딩 (완성출력버전)
with open("apple_ex.jpg", "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpg;base64,{image_data}"
            }
        },
        {
            "type": "text",
            "text": "이 이미지에 대해 설명해줘"
        }
    ]
)

response = llm.invoke([message])

print("=== 생각 과정 ===")
print(response.additional_kwargs.get('reasoning_content', ''))

print("\n=== 최종 답변 ===")
print(response.content)

=== 생각 과정 ===
이 사용자는 제공된 이미지에 대한 설명을 요청했습니다. 이미지의 주요 요소들을 분석하고, 이를 바탕으로 자연스러운 한국어 설명을 작성해야 합니다.

1.  **주인공:** 젊은 여성 (아시아인 혹은 중동계 혈통으로 보임). 긴 검은 머리카락, 흰색 반팔 티셔츠를 입고 있음.
2.  **동작:** 양손에 각각 빨간 사과를 들고 있음.
3.  **표정:** 미소를 짓고 있으며, 손에 든 사과들을 내려다보고 있음. 선택을 고민하거나 사과를 비교하는 듯한 분위기.
4.  **배경:** 단색의 주황색 배경.
5.  **분위기:** 밝고 건강해 보임. 과일이나 건강, 선택에 대한 이미지로 보임.

이 요소들을 조합하여 설명을 구성합니다.

*   **개요:** 젊은 여성이 빨간 사과 두 개를 들고 있는 사진입니다.
*   **상세 묘사:**
    *   **인물:** 긴 검은 머리카락을 내린 젊은 여성입니다. 흰색 상의를 입고 있습니다.
    *   **동작:** 양손에 각각 빨간 사과 하나씩 들고 있습니다.
    *   **표정:** 손에 든 사과들을 내려다보며 미소를 짓고 있습니다. 마치 두 사과 중 하나를 고르거나 비교하는 듯한 모습입니다.
    *   **배경:** 단색의 주황색 배경을 사용하여 인물과 사과가 돋보이게 합니다.
*   **의미/분위기:** 건강, 신선한 과일, 선택, 결정 등의 이미지를 줍니다.

이제 이 내용을 한국어로 자연스럽게 정리하여 답변을 생성합니다.

=== 최종 답변 ===
이 이미지는 밝고 활기찬 분위기를 가진 사진입니다. 주요 내용은 다음과 같습니다:

1.  **인물:** 긴 검은 머리카락을 내린 젊은 여성이 중앙에 서 있습니다. 흰색 반팔 티셔츠를 입고 있습니다.
2.  **동작:** 그녀의 양손에는 각각 빨간 사과 하나씩 들려 있습니다.
3.  **표정:** 그녀는 손에 든 사과들을 내려다보며 미소를 짓고 있습니다. 마치 두 사과 중 더 좋은 것을 고르거나, 두 사과를 비교하는 듯한 즐거운 표정

In [4]:
# 스트리밍 버전
with open("apple_ex.jpg", "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpg;base64,{image_data}"
            }
        },
        {
            "type": "text",
            "text": "이 이미지에 대해 설명해줘"
        }
    ]
)

reasoning_content = ""
final_content = ""

print("=== 생각 과정 ===")
for chunk in llm.stream([message]):
    # 생각 과정
    reasoning = chunk.additional_kwargs.get('reasoning_content', '')
    if reasoning:
        print(reasoning, end="", flush=True)
        reasoning_content += reasoning

    # 최종 답변
    if chunk.content:
        if not final_content:  # 답변 시작 시점에 헤더 출력
            print("\n\n=== 최종 답변 ===")
        print(chunk.content, end="", flush=True)
        final_content += chunk.content

print()  # 마지막 줄바꿈

=== 생각 과정 ===
이 사용자는 제공된 이미지에 대한 설명을 요청했습니다. 이미지의 주요 요소들을 분석하고, 이를 바탕으로 자연스러운 한국어 설명을 작성해야 합니다.

1.  **주인공:** 젊은 여성 (아시아인 혹은 중동계 혈통으로 보임). 긴 검은 머리카락, 흰색 반팔 티셔츠를 입고 있음.
2.  **동작:** 양손에 각각 빨간 사과를 들고 있음.
3.  **표정:** 미소를 짓고 있으며, 손에 든 사과들을 내려다보고 있음. 선택을 고민하거나 사과를 비교하는 듯한 분위기.
4.  **배경:** 단색의 주황색 배경.
5.  **분위기:** 밝고 건강해 보임. 과일이나 건강, 선택에 대한 이미지로 보임.

이 요소들을 조합하여 설명을 구성합니다.

*   **개요:** 젊은 여성이 빨간 사과 두 개를 들고 있는 사진입니다.
*   **상세 묘사:**
    *   **인물:** 긴 검은 머리카락을 내린 젊은 여성입니다. 흰색 상의를 입고 있습니다.
    *   **동작:** 양손에 각각 빨간 사과 하나씩 들고 있습니다.
    *   **표정:** 손에 든 사과들을 내려다보며 미소를 짓고 있습니다. 마치 두 사과 중 하나를 고르거나 비교하는 듯한 모습입니다.
    *   **배경:** 단색의 주황색 배경을 사용하여 인물과 사과가 돋보이게 합니다.
*   **의미/분위기:** 건강, 신선한 과일, 선택, 결정 등의 이미지를 줍니다.

이제 이 내용을 한국어로 자연스럽게 정리하여 답변을 생성합니다.

=== 최종 답변 ===
이 이미지는 밝고 활기찬 분위기를 가진 사진입니다. 주요 내용은 다음과 같습니다:

1.  **인물:** 긴 검은 머리카락을 내린 젊은 여성이 중앙에 서 있습니다. 흰색 반팔 티셔츠를 입고 있습니다.
2.  **동작:** 그녀의 양손에는 각각 빨간 사과 하나씩 들려 있습니다.
3.  **표정:** 그녀는 손에 든 사과들을 내려다보며 미소를 짓고 있습니다. 마치 두 사과 중 더 좋은 것을 고르거나, 두 사과를 비교하는 듯한 즐거운 표정

In [5]:
template = """
    너는 친절한 어시스턴트야. 다음 질문에 친절하게 답변해.
    질문 : {input}
    대답 :
"""

prompt = PromptTemplate.from_template(template)

In [6]:
chain = prompt | llm | StrOutputParser()

In [7]:
while True:
    query = input("\n질문을 입력하세요 (종료하려면 'quit' 입력): ")
    if query.strip().lower() in ['quit', 'exit']:
        print('대화를 종료합니다.')
        break

    answer = chain.invoke({"input": query})
    print("\n질문:\n", query)
    print("\n답변:\n", answer)

대화를 종료합니다.


이 이미지는 밝고 활기찬 분위기를 가진 사진입니다. 주요 내용은 다음과 같습니다:

1.  **인물:** 긴 검은 머리카락을 내린 젊은 여성이 중앙에 서 있습니다. 흰색 반팔 티셔츠를 입고 있습니다.
2.  **동작:** 그녀의 양손에는 각각 빨간 사과 하나씩 들려 있습니다.
3.  **표정:** 그녀는 손에 든 사과들을 내려다보며 미소를 짓고 있습니다. 마치 두 사과 중 더 좋은 것을 고르거나 비교하는 듯한 친근하고 즐거운 표정입니다.
4.  **배경:** 인물과 사과가 돋보이도록 단색의 주황색 배경을 사용했습니다.

전체적으로 건강, 신선한 과일, 혹은 선택에 대한 긍정적인 이미지를 전달하는 사진으로 보입니다.
